# 📘 Notebook 14: Proximal Policy Optimisation

### Course: *From Random Moves to Winning Agents: Reinforcement Learning with Games*
**Instructor:** Ioannis Tsioulis

*Module D: Policy-Based Learning · Notebook 3 of 4 in this module · (14 of 18 overall)*

---

## 🎯 Learning objectives

By the end of this notebook you will be able to:

- Explain why a policy-gradient update that is **too large** can ruin an agent, and why collected data can normally be used only once
- Define the **probability ratio** between the new and the old policy and use it to reuse data
- Write the **clipped objective** of PPO and explain what the clipping does for positive and for negative advantages
- Implement **PPO** on top of the actor-critic of Notebook 13
- Show by **ablation** that the clipping is what makes repeated updates safe
- Train an agent with the **Stable-Baselines3** library, knowing what happens inside

> **Prerequisites:** Notebook 13. This notebook reuses its two networks, its `collect` function and its advantage estimator without change.
>
> 🔭 **Why this notebook matters later:** PPO has been one of the most widely used reinforcement learning algorithms of recent years. It trained the Pokémon agent of Notebook 11, and it is the default choice in most libraries. Notebook 15 adapts it to continuous actions and Notebook 17 uses it against opponents.

> 📦 **Libraries used in this notebook.**
>
> - **PyTorch**, **Gymnasium** (with Box2D) and **imageio**.
> - **Stable-Baselines3** is a library of carefully tested implementations of reinforcement learning algorithms in PyTorch. We use it in the last section.
>
> No GPU is needed. The CartPole comparison takes a few minutes, and so does the lander.

In [ ]:
%pip install -q swig
%pip install -q "gymnasium[classic-control,box2d]>=1.3" stable-baselines3 imageio imageio-ffmpeg torch

In [ ]:
import base64
import time
import numpy as np
import matplotlib.pyplot as plt
import imageio
import gymnasium as gym
import torch
import torch.nn as nn
from IPython.display import HTML, display

torch.set_num_threads(2)


def show_video(frames, fps=30, width=420):
    """Play a list of pictures (RGB arrays) as a video inside the notebook."""
    imageio.mimsave("episode.mp4", frames, fps=fps, macro_block_size=1)
    data = base64.b64encode(open("episode.mp4", "rb").read()).decode()
    display(HTML(f'<video width="{width}" controls autoplay loop src="data:video/mp4;base64,{data}"></video>'))


def network(n_inputs, n_outputs, hidden=64):
    return nn.Sequential(nn.Linear(n_inputs, hidden), nn.Tanh(),
                         nn.Linear(hidden, hidden), nn.Tanh(),
                         nn.Linear(hidden, n_outputs))

## 1. Two problems of policy gradients

Notebook 13 ended with an agent that had a quiet gradient and was still unreliable. Two problems were left.

**Problem one: a step can be too large.** In supervised learning a bad update is corrected by the next batch, because the data stay the same. Here the policy **produces** its own data. A step that goes too far gives a worse policy, the worse policy collects worse experience, and the next update is computed from that. Exercise 4 of Notebook 12 showed an agent that did well and then destroyed itself.

**Problem two: data are used once.** The policy gradient is an expectation over the **current** policy. After one update the policy has changed, so the batch that was just collected no longer comes from it, and strictly speaking it has to be thrown away. Collecting experience is the expensive part, so this is wasteful.

The two problems are linked. If we knew that an update kept the new policy **close** to the one that collected the data, the data would still be nearly valid, and we could use them again.

### Intuition first
A hiker in thick fog can see the ground for a few metres around. A map of the slope under their feet is trustworthy for a few steps and worthless a kilometre away. The sensible rule is: take a few steps in the downhill direction, look at the ground again, and repeat. Never take a long stride on the strength of a local look.

A batch of experience is that local look. **Proximal** policy optimisation means: improve the policy as much as the batch allows, while staying in the neighbourhood (the proximity) where the batch can be trusted.

## 2. Reusing data: the probability ratio

### Formal definition
Suppose the batch was collected by the policy $\pi_\text{old}$, and we now consider a changed policy $\pi$. For each recorded step we can compute how the change affects the action that was taken:

$$r_t = \frac{\pi(A_t \mid S_t)}{\pi_\text{old}(A_t \mid S_t)}$$

This **probability ratio** is 1 if the new policy is as likely to take that action as the old one, above 1 if it has become more likely, and below 1 if less likely.

With it, the quantity that the policy gradient improves can be written for the new policy using data from the old one:

$$L(\boldsymbol\theta) = \mathbb{E}\big[\, r_t\,\hat A_t \,\big]$$

Read it as an instruction: make actions with positive advantage more likely ($r_t > 1$ where $\hat A_t > 0$) and actions with negative advantage less likely. At the starting point, where $r_t = 1$ everywhere, its gradient is exactly the A2C gradient of Notebook 13.

### Why it is not enough
Maximising $L$ for many steps on the same batch is tempting, and it is the mistake of the hiker with the long stride. Nothing stops the optimiser from pushing some ratio to 5 or to 0.01 because a handful of samples said that an action was good or bad. The estimate of $L$ is only reliable while the ratios stay near 1.

## 3. The clipped objective

### Formal definition
PPO (John Schulman and colleagues at OpenAI, 2017) changes the objective so that there is **nothing to gain** from moving a ratio far from 1. With a small number $\varepsilon$, typically 0.2:

$$L^\text{clip}(\boldsymbol\theta) = \mathbb{E}\Big[\min\big(\; r_t\,\hat A_t,\;\; \text{clip}(r_t,\, 1-\varepsilon,\, 1+\varepsilon)\,\hat A_t \;\big)\Big]$$

`clip` cuts the ratio off at $1-\varepsilon$ and $1+\varepsilon$. The `min` then takes the more pessimistic of the clipped and the unclipped value. A plot shows what that does.

In [ ]:
ratio = np.linspace(0, 2, 201)
epsilon = 0.2
clipped_ratio = np.clip(ratio, 1 - epsilon, 1 + epsilon)

fig, axes = plt.subplots(1, 2, figsize=(11, 4), sharey=True)
for ax, advantage in zip(axes, (1.0, -1.0)):
    ax.plot(ratio, ratio * advantage, linestyle="--", label="unclipped: ratio × advantage")
    ax.plot(ratio, np.minimum(ratio * advantage, clipped_ratio * advantage), linewidth=2.5, label="PPO objective")
    ax.axvline(1, color="gray", linestyle=":")
    ax.set_title(f"advantage = {advantage:+.0f}  (the action was {'good' if advantage > 0 else 'bad'})")
    ax.set_xlabel("probability ratio  (new policy / old policy)"); ax.grid(True); ax.legend()
axes[0].set_ylabel("contribution to the objective")
plt.show()

Follow the thick line in each panel, starting from the dotted line at ratio 1, where every update begins.

- **A good action (left).** Raising its probability increases the objective, but only up to a ratio of 1.2. Beyond that the line is flat: making the action still more likely earns nothing, so the gradient is zero and the optimiser stops pushing.
- **A bad action (right).** Lowering its probability increases the objective, down to a ratio of 0.8, and then the line is flat again.
- **In the wrong direction** the line is **not** flat. If an update has made a bad action more likely (right panel, ratio above 1), the objective keeps falling, so there is always a gradient pulling it back. The `min` is what guarantees this: the objective is cut off only where the cut makes it smaller.

The result is an objective that rewards a modest step in the right direction for each sample and offers no reward for going further. An optimiser can therefore run over the same batch **several times** without being able to wreck the policy.

## 4. The algorithm

PPO is A2C with a different update:

1. **Collect** a rollout with the current policy, in several environments in parallel (Notebook 13).
2. Compute the **advantages** with GAE, and remember the log-probability that the policy gave to each action taken. These define $\pi_\text{old}$.
3. Run several **epochs** over the rollout in shuffled **minibatches**, each time taking a gradient step on the clipped objective (plus the critic's loss and the entropy bonus, as before).
4. Throw the rollout away and go back to step 1.

The next cell repeats `collect` and `compute_gae` from Notebook 13 unchanged. Run it. You do not need to read it again.

In [ ]:
def collect(envs, states, actor, critic, n_steps, finished_returns, running_returns):
    """Play n_steps in every environment. Returns the recorded rollout and the new current states."""
    n_envs = len(envs)
    shape = (n_steps, n_envs)
    rollout = dict(states=np.zeros(shape + states.shape[1:], dtype=np.float32),
                   actions=np.zeros(shape, dtype=np.int64), rewards=np.zeros(shape, dtype=np.float32),
                   terminated=np.zeros(shape, dtype=np.float32), done=np.zeros(shape, dtype=np.float32),
                   values=np.zeros(shape, dtype=np.float32), next_values=np.zeros(shape, dtype=np.float32))
    for t in range(n_steps):
        with torch.no_grad():
            batch = torch.as_tensor(states)
            actions = torch.distributions.Categorical(logits=actor(batch)).sample().numpy()
            rollout["values"][t] = critic(batch).squeeze(1).numpy()
        rollout["states"][t], rollout["actions"][t] = states, actions
        next_states = np.zeros_like(states)
        for i, env in enumerate(envs):
            state, reward, terminated, truncated, info = env.step(int(actions[i]))
            rollout["rewards"][t, i] = reward
            rollout["terminated"][t, i] = terminated
            rollout["done"][t, i] = terminated or truncated
            next_states[i] = state                              # the state the action led to
            running_returns[i] += reward
            if terminated or truncated:
                finished_returns.append(running_returns[i])
                running_returns[i] = 0.0
                state, info = env.reset()
            states[i] = state                                   # where this environment continues
        with torch.no_grad():
            rollout["next_values"][t] = critic(torch.as_tensor(next_states)).squeeze(1).numpy()
    return rollout, states


def compute_gae(rollout, gamma, lam):
    """GAE for a rollout of shape (steps, environments), respecting the ends of episodes."""
    advantages = np.zeros_like(rollout["rewards"])
    running = np.zeros(advantages.shape[1])
    for t in reversed(range(len(advantages))):
        delta = (rollout["rewards"][t] + gamma * (1 - rollout["terminated"][t]) * rollout["next_values"][t]
                 - rollout["values"][t])
        running = delta + gamma * lam * (1 - rollout["done"][t]) * running
        advantages[t] = running
    return advantages

In [ ]:
def ppo(env_name, total_steps, n_envs=8, n_steps=256, epochs=4, batch_size=64, clip=0.2, gamma=0.99, lam=0.95,
        learning_rate=3e-4, entropy_coef=0.01, seed=0, log=None):
    """Proximal policy optimisation. With clip=None the objective is the unclipped ratio x advantage.
    Returns the actor, the critic, the return of every finished episode, and one diagnostic per update."""
    torch.manual_seed(seed)
    rng = np.random.default_rng(seed)
    envs = [gym.make(env_name) for i in range(n_envs)]
    states = np.array([env.reset(seed=seed * 1000 + i)[0] for i, env in enumerate(envs)], dtype=np.float32)
    state_size, n_actions = states.shape[1], envs[0].action_space.n
    actor, critic = network(state_size, n_actions), network(state_size, 1)
    parameters = list(actor.parameters()) + list(critic.parameters())
    optimiser = torch.optim.Adam(parameters, lr=learning_rate)
    finished_returns, running_returns, steps, updates, moved = [], np.zeros(n_envs), 0, 0, []

    while steps < total_steps:
        # 1. collect
        rollout, states = collect(envs, states, actor, critic, n_steps, finished_returns, running_returns)
        steps += n_steps * n_envs

        # 2. advantages, and the old log-probabilities
        advantages = compute_gae(rollout, gamma, lam)
        s = torch.as_tensor(rollout["states"].reshape(-1, state_size))
        a = torch.as_tensor(rollout["actions"].reshape(-1))
        advantage = torch.as_tensor(advantages.reshape(-1))
        target = torch.as_tensor((advantages + rollout["values"]).reshape(-1))
        with torch.no_grad():
            old_log_probability = torch.distributions.Categorical(logits=actor(s)).log_prob(a)

        # 3. several epochs of minibatch updates on the same rollout
        for epoch in range(epochs):
            order = rng.permutation(len(s))
            for start in range(0, len(s), batch_size):
                index = torch.as_tensor(order[start:start + batch_size])
                distribution = torch.distributions.Categorical(logits=actor(s[index]))
                ratio = torch.exp(distribution.log_prob(a[index]) - old_log_probability[index])
                adv = advantage[index]
                adv = (adv - adv.mean()) / (adv.std() + 1e-8)
                if clip is None:
                    actor_loss = -(ratio * adv).mean()
                else:
                    clipped = torch.clamp(ratio, 1 - clip, 1 + clip)
                    actor_loss = -torch.min(ratio * adv, clipped * adv).mean()
                critic_loss = (critic(s[index]).squeeze(1) - target[index]).pow(2).mean()
                loss = actor_loss + 0.5 * critic_loss - entropy_coef * distribution.entropy().mean()
                optimiser.zero_grad()
                loss.backward()
                nn.utils.clip_grad_norm_(parameters, 0.5)
                optimiser.step()

        # diagnostic: for which share of the samples did the probability change by more than 20%?
        with torch.no_grad():
            final_ratio = torch.exp(torch.distributions.Categorical(logits=actor(s)).log_prob(a) - old_log_probability)
        moved.append(((final_ratio - 1).abs() > 0.2).float().mean().item())

        updates += 1
        if log and updates % log == 0 and finished_returns:
            print(f"step {steps:>7,}   episodes {len(finished_returns):>5}   "
                  f"average return of the last 50: {np.mean(finished_returns[-50:]):7.1f}")
    return actor, critic, np.array(finished_returns), np.array(moved)

The ratio is computed as `exp(new log-probability − old log-probability)`, which is the same as dividing the probabilities and numerically safer.

Compare the update with A2C in Notebook 13. There, each rollout of 256 steps (8 environments × 32 steps) gave **one** gradient step. Here a rollout of 2,048 steps is cut into minibatches of 64 and passed over `epochs` times: with 4 epochs that is 128 gradient steps per rollout, each of them held on a leash by the clipping.

## 5. Does the clipping matter?

An ablation, as in Notebook 09. We use each rollout **ten times** (`epochs=10`), which is a lot of reuse, once with the clipped objective and once without (`clip=None`). Three seeds each, 60,000 steps of CartPole. This takes a few minutes.

In [ ]:
def smooth(values, window=20):
    return np.convolve(values, np.ones(window) / window, mode="valid")


with_clip = [ppo("CartPole-v1", 60000, epochs=10, seed=seed) for seed in range(3)]
without_clip = [ppo("CartPole-v1", 60000, epochs=10, clip=None, seed=seed) for seed in range(3)]

fig, axes = plt.subplots(1, 2, figsize=(12, 4), sharey=True)
for ax, title, group in ((axes[0], "with clipping (PPO)", with_clip), (axes[1], "without clipping", without_clip)):
    for seed, (actor, critic, returns, moved) in enumerate(group):
        ax.plot(smooth(returns), label=f"seed {seed}")
    ax.axhline(500, color="gray", linestyle=":")
    ax.set_title(title); ax.set_xlabel("episode"); ax.grid(True); ax.legend()
axes[0].set_ylabel("steps the pole stayed up (average of 20)")
plt.show()

for title, group in (("with clipping", with_clip), ("without clipping", without_clip)):
    print(title)
    for seed, (actor, critic, returns, moved) in enumerate(group):
        curve = smooth(returns)
        print(f"  seed {seed}: best 20-episode average {curve.max():4.0f},  last 20 episodes {returns[-20:].mean():4.0f},  "
              f"samples whose probability moved by more than 20% per update: {100 * moved.mean():2.0f}%")

Read the last column first. With clipping, one round of updates changes the probability of the chosen action by more than 20% for only a few samples in a hundred. Without it, that happens for about half of them: the policy after the update is a different policy from the one that collected the data.

Then compare "best" with "last". The clipped agents reach the top and **stay** there. Without clipping, agents reach a high score and lose it again, the pattern we have now met in Notebooks 08, 12 and 13.

The clipping costs one line of code. It is the difference between an agent that can be trusted to keep what it has learned and one that cannot.

## 6. Lunar Lander

A2C left us with one run of 180 and others far below (Notebook 13). Now PPO, on the same 600,000 steps. Two settings are changed from CartPole: a discount factor very close to 1 and a higher $\lambda$, because landing is rewarded only at the very end of a long episode and the agent has to look far ahead.

**This cell takes a few minutes.**

In [ ]:
start = time.time()
lander_actor, lander_critic, lander_returns, lander_moved = ppo(
    "LunarLander-v3", 600000, gamma=0.999, lam=0.98, log=20)
print(f"\ntraining took {time.time() - start:.0f} seconds")

plt.figure(figsize=(8, 4))
plt.plot(lander_returns, alpha=0.25, label="each episode")
plt.plot(np.arange(49, len(lander_returns)), smooth(lander_returns, 50), label="average of 50")
plt.axhline(200, color="green", linestyle="--", label="solved")
plt.title("PPO on Lunar Lander")
plt.xlabel("episode"); plt.ylabel("return"); plt.grid(True); plt.legend()
plt.show()

print(f"average return of the last 50 training episodes: {lander_returns[-50:].mean():.0f}")

In [ ]:
def record(env, actor, seed=0):
    state, info = env.reset(seed=seed)
    frames, total = [env.render()], 0.0
    terminated = truncated = False
    while not (terminated or truncated):
        with torch.no_grad():
            action = int(actor(torch.as_tensor(state, dtype=torch.float32)).argmax())     # the most probable action
        state, reward, terminated, truncated, info = env.step(action)
        total += reward
        frames.append(env.render())
    return frames, total


frames, total = record(gym.make("LunarLander-v3", render_mode="rgb_array"), lander_actor, seed=0)
print(f"trained agent: return {total:.0f}")
show_video(frames[::2], fps=25)

The curve climbs without the long relapses of the A2C runs. One run proves little, as this course keeps saying, so here are the runs we made while preparing the notebook: 600,000 steps each, average return of the last 50 episodes.

| Agent | Runs |
|---|---|
| A2C (Notebook 13) | 180, −33, −110 |
| PPO without clipping (`clip=None`) | 136, 139 |
| **PPO** | **238, 234, 199** |

PPO is the only one of the three whose runs all end near or above the "solved" mark of 200. It uses the same networks, the same advantage estimates and the same amount of experience as the others. The difference is in how far each update is allowed to go, and in how much use is made of each batch as a result.

> ⚠️ **Common pitfalls**
>
> - **Recomputing the "old" log-probabilities during the epochs.** They must be computed once, before the first update, and kept fixed. Otherwise every ratio is 1 and nothing is clipped.
> - **Too many epochs.** Clipping makes reuse safe within limits, not without limit. Watch a diagnostic such as the share of samples that moved.
> - **Forgetting to normalise the advantages.** The clipping range is fixed, so the scale of the advantages decides how hard the optimiser pulls.
> - **Copying hyperparameters between tasks.** The lander needed a different discount factor from CartPole to reach 200.

## 7. The same thing from a library

You have now written PPO yourself. For real projects you would normally use a tested implementation, and **Stable-Baselines3** is the most widely used one in PyTorch. Three lines train an agent:

In [ ]:
from stable_baselines3 import PPO
from stable_baselines3.common.evaluation import evaluate_policy

start = time.time()
model = PPO("MlpPolicy", "CartPole-v1", seed=0, verbose=0)
model.learn(total_timesteps=50000)
print(f"training took {time.time() - start:.0f} seconds")

mean_return, spread = evaluate_policy(model, gym.make("CartPole-v1"), n_eval_episodes=20)
print(f"average return over 20 episodes: {mean_return:.0f} (standard deviation {spread:.0f})")

Everything you see in those lines has a counterpart in our function:

| Stable-Baselines3 | Our `ppo` | Default in the library |
|---|---|---|
| `"MlpPolicy"` | `network(...)` for actor and critic | two hidden layers of 64 units, tanh |
| `n_steps` | `n_steps` | 2,048 steps per environment |
| `n_epochs`, `batch_size` | `epochs`, `batch_size` | 10 epochs, minibatches of 64 |
| `clip_range` | `clip` | 0.2 |
| `gamma`, `gae_lambda` | `gamma`, `lam` | 0.99 and 0.95 |
| `ent_coef` | `entropy_coef` | 0 |

In [ ]:
print("clip range:", model.clip_range(1.0), "  epochs:", model.n_epochs, "  gamma:", model.gamma, "  GAE lambda:", model.gae_lambda)

video_env = gym.make("CartPole-v1", render_mode="rgb_array")
state, info = video_env.reset(seed=0)
frames = [video_env.render()]
terminated = truncated = False
while not (terminated or truncated):
    action, _ = model.predict(state, deterministic=True)
    state, reward, terminated, truncated, info = video_env.step(int(action))
    frames.append(video_env.render())
print(f"the pole stayed up for {len(frames) - 1} steps")
show_video(frames[::3], fps=20)

model.save("ppo_cartpole")              # PPO.load("ppo_cartpole") brings it back

A library saves time and avoids bugs. It does not remove the need to understand the algorithm: every row of the table above is a decision that you will have to make, or accept by default, for each new problem.

### Why PPO became the default

- It is **simple**: a policy gradient with one changed line in the objective.
- It is **robust**: the same settings work passably on many problems, which is rare in this field.
- It **scales**: experience can be collected by as many parallel environments as there are processors, as in the Pokémon project of Notebook 11, which ran 16 emulators.

It has been used at very large scale, for example by OpenAI for an agent that reached the level of professional teams in the video game Dota 2, and to fine-tune language models from human feedback. What it is not is frugal. Like all on-policy methods it throws its data away after a few passes. When each step of experience is expensive, off-policy methods with a replay buffer use fewer of them, and the next notebook meets one.

---
## ✏️ Exercises

The exercises use `ppo`, `smooth`, `record` and the results of Sections 5 and 6.

### Exercise 1 (The clipped objective by hand)
With $\varepsilon = 0.2$, compute the contribution $\min(r\,\hat A,\ \text{clip}(r, 0.8, 1.2)\,\hat A)$ for these four samples, on paper and then in Python: (a) $r = 1.5$, $\hat A = +2$; (b) $r = 0.5$, $\hat A = +2$; (c) $r = 1.5$, $\hat A = -2$; (d) $r = 0.5$, $\hat A = -2$. In which of the four cases is the gradient with respect to the ratio zero?

In [ ]:
# Your solution here:


<details><summary>💡 Show solution</summary>

```python
def contribution(ratio, advantage, epsilon=0.2):
    return min(ratio * advantage, float(np.clip(ratio, 1 - epsilon, 1 + epsilon)) * advantage)

for label, ratio, advantage in (("a", 1.5, 2), ("b", 0.5, 2), ("c", 1.5, -2), ("d", 0.5, -2)):
    flat = contribution(ratio, advantage) == contribution(ratio + 0.01, advantage)
    print(f"({label}) ratio {ratio}, advantage {advantage:+d}: contribution {contribution(ratio, advantage):+.1f}   "
          f"gradient zero: {flat}")
```

*In (a) a good action has already been made 50% more likely: the clipped value 2.4 is the smaller one, and the objective is flat. In (d) a bad action has already been made much less likely: flat again. In (b) and (c) the policy has moved the **wrong** way, the unclipped value is the smaller one, and the gradient is fully present to bring the ratio back.*
</details>

### Exercise 2 (How often can a batch be used?)
Train PPO on CartPole for 60,000 steps with `epochs` = 1, 4 and 10 (seed 0 and 1 each, with clipping). Print the average return over the whole training, which measures how fast the agent learned, and the share of samples that moved by more than 20%. What does one epoch correspond to among the algorithms you know?

In [ ]:
# Your solution here:


<details><summary>💡 Show solution</summary>

```python
for epochs in (1, 4, 10):
    for seed in range(2):
        if epochs == 10:
            actor, critic, returns, moved = with_clip[seed]             # already trained in Section 5
        else:
            actor, critic, returns, moved = ppo("CartPole-v1", 60000, epochs=epochs, seed=seed)
        print(f"epochs = {epochs:>2}, seed {seed}: average return over the whole training {returns.mean():4.0f},  "
              f"last 20 episodes {returns[-20:].mean():4.0f},  moved {100 * moved.mean():2.0f}%")
```

*More passes over each rollout mean faster learning from the same experience. With a single epoch every sample is used once, as in A2C, only with larger rollouts and smaller minibatches, and the agent is well behind after 60,000 steps. The share of samples that moved stays small in all three cases: that is the clipping at work, and it is what makes the extra passes safe.*
</details>

### Exercise 3 (The width of the clip)
Train PPO on CartPole for 60,000 steps with `epochs=10` and `clip` = 0.05, 0.2 and 0.8, for seeds 0 to 2. Print the best 20-episode average, the last 20 episodes, and the share of samples that moved by more than 20%. What happens at each extreme?

In [ ]:
# Your solution here:


<details><summary>💡 Show solution</summary>

```python
for clip in (0.05, 0.2, 0.8):
    for seed in range(3):
        if clip == 0.2:
            actor, critic, returns, moved = with_clip[seed]
        else:
            actor, critic, returns, moved = ppo("CartPole-v1", 60000, epochs=10, clip=clip, seed=seed)
        print(f"clip = {clip:<4} seed {seed}: best {smooth(returns).max():4.0f},  last 20 episodes {returns[-20:].mean():4.0f},  "
              f"moved {100 * moved.mean():2.0f}%")
```

*A very narrow clip keeps every update tiny. That is safe and, on harder problems, slow. A very wide clip allows almost any change, and the agent behaves much like the unclipped one of Section 5: it can reach the top and fall off again. The usual value of 0.2 is a compromise that has held up on a wide range of problems.*
</details>

### Exercise 4 (The lander without the leash)
Train the lander with the settings of Section 6 but `clip=None`. **This takes as long as the training in the notebook.** Compare the final average return and the share of samples that moved with the clipped run, and compare your number with the table in Section 6.

In [ ]:
# Your solution here:


<details><summary>💡 Show solution</summary>

```python
actor_free, critic_free, returns_free, moved_free = ppo("LunarLander-v3", 600000, gamma=0.999, lam=0.98, clip=None, log=40)

print(f"\nwith clipping:    last 50 episodes {lander_returns[-50:].mean():6.0f},  moved {100 * lander_moved.mean():2.0f}%")
print(f"without clipping: last 50 episodes {returns_free[-50:].mean():6.0f},  moved {100 * moved_free.mean():2.0f}%")
```

*Without clipping the agent still learns to slow its fall, and it does not collapse as dramatically as on CartPole with ten epochs, since this run uses four. But it ends well short of the clipped agent. Each rollout changes the policy much more than it should, so the advantages computed from that rollout are being applied to a policy they no longer describe.*
</details>

### Exercise 5 (Our PPO against the library (a little harder))
Train Stable-Baselines3's PPO on CartPole for 60,000 steps with settings that match our Section 5 run as closely as the table in Section 7 allows: `n_steps=256`, `n_epochs=10`, `batch_size=64`, `ent_coef=0.01`, and 8 parallel environments, which you get by passing `make_vec_env("CartPole-v1", n_envs=8, seed=0)` from `stable_baselines3.common.env_util` in place of the name. Evaluate it over 20 episodes with `evaluate_policy`. Evaluate our seed-0 agent of Section 5 in the same way, by playing 20 episodes with its most probable action. Do the two implementations agree?

In [ ]:
# Your solution here:


<details><summary>💡 Show solution</summary>

```python
from stable_baselines3.common.env_util import make_vec_env

library_model = PPO("MlpPolicy", make_vec_env("CartPole-v1", n_envs=8, seed=0), n_steps=256, n_epochs=10,
                    batch_size=64, ent_coef=0.01, seed=0, verbose=0)
library_model.learn(total_timesteps=60000)
library_score, spread = evaluate_policy(library_model, gym.make("CartPole-v1"), n_eval_episodes=20)

our_actor = with_clip[0][0]
test_env = gym.make("CartPole-v1")
our_scores = []
for episode in range(20):
    state, info = test_env.reset(seed=episode)
    total, terminated, truncated = 0.0, False, False
    while not (terminated or truncated):
        with torch.no_grad():
            action = int(our_actor(torch.as_tensor(state)).argmax())
        state, reward, terminated, truncated, info = test_env.step(action)
        total += reward
    our_scores.append(total)

print(f"Stable-Baselines3: {library_score:.0f}")
print(f"our PPO:           {np.mean(our_scores):.0f}")
```

*Two independent implementations of the same algorithm, with matching settings, reach the same result on this task. That is a useful check on our own code. It is also how results in this field should be treated in general: a number means more once somebody else's implementation has reproduced it.*
</details>

## 🔑 Key takeaways

- A policy-gradient step that is too large gives a worse policy, which collects worse data: errors **compound**.
- The **probability ratio** $\pi / \pi_\text{old}$ lets a batch collected by one policy be used to improve a slightly different one.
- PPO's **clipped objective** removes any reward for moving a ratio beyond $1 \pm \varepsilon$, while keeping the full gradient for mistakes.
- Clipping makes it safe to run **several epochs** over each rollout, so PPO learns more from the same experience than A2C.
- In our runs PPO was the only policy-gradient agent whose Lunar Lander runs **all** ended near or above 200.
- **Stable-Baselines3** offers the same algorithm ready-made. Its arguments are the decisions you made yourself in this notebook.

---
**Next:** *Notebook 15: Continuous Actions*, where the agent turns a dial where it used to press buttons.

---
*© Ioannis Tsioulis. *From Random Moves to Winning Agents: Reinforcement Learning with Games*. Prepared for educational use.*